# 破解 CRC32：从一个哈希值出发，反推出原始的输入数据

## 这一章讲什么

CRC32 是用于检查数据完整性的哈希，但它在密码学意义上并不安全：输出一旦摆在眼前，就可以算出一个满足条件的输入来。这一章把 CRC32 的计算原样写进 Z3，再由求解器造出符合条件的数据。

- 把 CRC32 的计算用 Z3 的 `BitVec` 一句一句照抄下来
- 以指定的字符串开头、只含英文字母、CRC32 正好等于给定值，这些都写成约束
- 剩下的工作交给 Z3，让它把符合条件的数据造出来

手头的工具只有两样：`BitVec` 的位运算(`>>`, `&`, `^`, `~`)与 `ZeroExt`(把位宽撑开)。

## 什么是 CRC32

CRC32(Cyclic Redundancy Check 32-bit)是 32 位的校验和，专门用来发现数据中出现的错误，网络通信与文件完整性检查都离不开它。

然而它不是密码学意义上安全的哈希：只要拿到一个输出，例如某个文件的校验值，就能反推出产生这个输出的输入。为什么这种算法不能用来保存密码，这一章请读者亲手试一遍，试过就明白了。

## CRC32 的计算方法

CRC32 究竟在计算什么，看下面这段 C 代码就清楚了。

```c
static uint32_t crc32(uint32_t crc, unsigned char *buf, size_t len)
{
    int k;

    crc = ~crc;
    while (len--) {
        crc ^= *buf++;
        for (k = 0; k < 8; k++)
            crc = crc & 1 ? (crc >> 1) ^ 0xedb88320 : crc >> 1;
    }
    return ~crc;
}
```

拆开来看，它只做四件事。

1. 先把初值按位取反(`crc = ~crc`)
2. 再把每个字节依次 XOR 进去
3. 然后重复 8 遍：右移 1 位，最低位是 1 就把多项式 `0xedb88320` XOR 上去
4. 最后再按位取反一次，然后返回

这套计算在 Python 中同样可以一行行写出来。换成 Z3 的 `BitVec` 之后，输入可以一直是变量，照样顺着算式往下走，不必先把具体的值填进去。

In [1]:
from z3 import *


def z3_crc32(data, crc=0):
    """返回 data（BitVec(8) 的列表）的 CRC32，形式为 Z3 的表达式"""
    crc = ~crc                      # 反转初始值
    for c in data:
        crc ^= ZeroExt(24, c)       # 把 8 位字节扩展到 32 位再 XOR
        for _ in range(8):
            crc = If(crc & 1 == BitVecVal(1, 32),
                     LShR(crc, 1) ^ 0xedb88320,
                     LShR(crc, 1))
    return ~crc                     # 最后反转

## 用到的 Z3 函数

| 函数与运算符 | 作用 |
|---|---|
| `ZeroExt(24, c)` | 把 8 位的值在高位补 0，撑成 32 位 |
| `BitVecVal(1, 32)` | 建立一个 32 位常数 1(位宽属于类型的一部分) |
| `LShR(crc, 1)` | 无符号右移，空出来的高位补 0 |
| `If(条件, A, B)` | 条件为真时取 A，为假时取 B |
| `~crc` | 按位取反 |
| `^`, `&` | XOR 与 AND |

这里使用 `LShR()` 而不使用 `>>`，是因为 `BitVec` 分为有符号与无符号两种，其右移的含义并不相同。要与 C 的 `unsigned` 走同一条路，右移就必须使用 `LShR()`，让空出来的高位补 0。

至于 `BitVecVal()`，它表示的是“值固定的 `BitVec`”，其值不随求解过程变化；与之相邻的 `simplify()` 负责化简表达式，所有值都确定下来的时候，其结果就是一个数字，表达式的形状也就消失了。

## 先与 `binascii.crc32()` 对一下

拿 Python 自带的 `binascii.crc32()` 比一比：同一个输入喂进去，两边算出的数相同，我们写的这段实现才算站得住。

In [2]:
from binascii import crc32

data = b'hello world!!'
print(crc32(data))

simplify(z3_crc32([BitVecVal(x, 8) for x in data]))

2909492980


2909492980

对上了。由于 `data` 的每个字节都是用 `BitVecVal(x, 8)` 这样的常数喂进去的，因此 Z3 的表达式一路化简到底，就是一个数值。

## 从 CRC32 反推数据

正题从这里开始，下面这几条条件交给 Z3，由它找出满足条件的数据。

- 长度是 `n` 字节
- 以 `head` 开头，以 `tail` 结尾
- 中间的字符全是英文字母(`a`〜`z`, `A`〜`Z`)
- CRC32 等于 `crc_code`

把数据的每个字节都设成 `BitVec(f'b{i}', 8)` 这样的变量以后，“CRC32 等于 `crc_code`”本身就成了一条约束，其中每个变量表示一个字节。

In [3]:
def is_between(b, start, end):
    """b 在 start 到 end 的字符编码范围内"""
    return And(b >= ord(start), b <= ord(end))


def starts_with(data, string):
    """data 以 string 开头"""
    return And([b == ord(ch) for b, ch in zip(data, string)])


def ends_with(data, string):
    """data 以 string 结尾"""
    return And([b == ord(ch) for b, ch in zip(data[::-1], string[::-1])])


def break_crc32(n, head, tail, crc_code):
    """找出满足条件的 n 字节数据"""
    data = [BitVec(f'b{i:02d}', 8) for i in range(n)]

    solver = Solver()
    solver.add(starts_with(data, head))
    solver.add(ends_with(data, tail))

    # 中间的字符只能是英文字母
    for b in data[len(head):-len(tail)]:
        solver.add(Or(is_between(b, 'a', 'z'), is_between(b, 'A', 'Z')))

    solver.add(z3_crc32(data) == crc_code)

    if solver.check() != sat:
        return None

    model = solver.model()
    return bytes([model[d].as_long() for d in data])


def test(n, head, tail, crc_code):
    res = break_crc32(n, head, tail, crc_code)
    if res is None:
        print('unsat（不存在满足条件的数据）')
    else:
        print(res, hex(crc32(res)))

## 执行示例

要找的是一段长度为 20 的字符串：以 `"hello "` 开头，以 `"!!"` 结尾，中间可以任意挑选，只要求算出来的 CRC32 恰好等于 `0x12345678`。

In [4]:
test(20, 'hello ', '!!', 0x12345678)

b'hello PWvJSuTYDVsZ!!' 0x12345678


输出末尾那个 `0x...` 就是生成出来的字符串的 CRC32，与指定的 `0x12345678` 对一对，两者应当相等。由于这里又用 `binascii.crc32()` 重算了一遍，因此 Z3 的答案到底对不对，一查便知。

## 太短就解不出来

字符数一少，能够自由挑选的字符也随之减少。这种情况下如果根本不存在满足条件的数据，Z3 只能返回 `unsat`。

In [5]:
test(13, 'hello ', '!!', 0x00000000)

unsat（不存在满足条件的数据）


`head` 占 6 个字符，`tail` 占 2 个，能自由挑选的中间部分只剩 5 个字符。注意，Z3 证明了这 5 个字符无论怎样排列，CRC32 都到不了 0。

下面把长度增加一个字节，再问它一次。

In [6]:
test(14, 'hello ', '!!', 0x00000000)

b'hello zXforl!!' 0x0


只不过又多给了一个自由字符，数据就找到了。

- 能够自由搜索的字符越多，满足条件的数据就越好找
- CRC32 的计算是线性的，条件原样写成约束即可，其中线性表示每一步都只有异或与移位
- 因此 CRC32 能用于检测篡改，却不能用于防止篡改

## 小结

- CRC32 的计算可以用 `BitVec` 的位运算一行行照抄
- 8 位的字节靠 `ZeroExt(24, c)` 撑到 32 位
- 无符号右移用 `LShR()`，常数用 `BitVecVal()`
- 开头、结尾、字符种类、CRC 值这些条件写成约束，Z3 就能把数据造出来
- 自由字符太少，结果就是 `unsat`，也就是被证明了确实不存在